# Local support-ticket routing with Laya

Use Laya's typed decision API to suggest a support queue for a short English ticket. This walkthrough uses the project's `TicketTriage` adapter, so its behavior matches the command-line example.

## What Laya does

Laya is a non-autoregressive System 1 decision model. It answers typed questions such as `choice`, `score`, and `noul`; it does not generate free-form text and is not a chat model. The initial example uses a small `choice` question for billing, technical, account, or other support.

The output is advisory only. Do not automatically issue refunds, close accounts, or take other consequential actions from this prediction.

## Prerequisites and setup

Run these commands from the project root. Install `uv` first if it is not already available. The lockfile installs the Laya SDK and the notebook kernel into the project environment.

```sh
uv sync
uv run --group dev python -m ipykernel install --user --name laya-local-ai-demo --display-name "Python (laya-local-ai-demo)"
```

In VS Code, select the `Python (laya-local-ai-demo)` kernel for this notebook.

In [11]:
import laya

print(f"Laya SDK version: {laya.__version__}")

Laya SDK version: 0.3.26


## First use and local execution

Installing the Python package does not download model weights. The first prediction downloads the selected checkpoint from Hugging Face, so that first run needs network access and enough disk space. Later inference runs locally from the cached weights; this walkthrough does not call a hosted inference API. Actual memory use depends on your machine and runtime.

In [8]:
from laya_local_ai_demo.triage import QUEUE_DESCRIPTIONS

for queue, description in QUEUE_DESCRIPTIONS.items():
    print(f"{queue}: {description}")

billing: Invoices, payments, duplicate charges, or refunds
technical: Product bugs, errors, outages, or integrations
account: Login, password, access, or account settings
other: A request that does not fit the other support queues


## Classify one synthetic ticket

The example asks Laya to choose among four distinct queues and prints both the suggestion and the checkpoint selected by the router. The prediction is not a confidence or correctness guarantee.

In [9]:
from dataclasses import asdict

from laya_local_ai_demo.triage import TicketTriage

ticket_text = "I was charged twice for my monthly subscription."
triage_result = TicketTriage().classify(ticket_text)

asdict(triage_result)

{'queue': 'billing', 'checkpoint': 'english'}

The result is a structured suggestion for a human to review. Laya's published benchmarks are not a substitute for evaluating your own ticket distribution; the model can be wrong, and this four-label demo does not establish production accuracy.

## Evaluate the tiny demo fixture

The repository includes eight synthetic, hand-labeled examples. The script reports their raw accuracy and individual outcomes. This is only a walkthrough check, not a representative benchmark or a general quality claim.

In [10]:
import subprocess
from pathlib import Path

evaluation_process = subprocess.run(
    ["uv", "run", "python", "scripts/evaluate_demo.py"],
    check=True,
    cwd=Path(".."),
)

{
  "correct": 7,
  "total": 8,
  "accuracy": 0.875,
  "results": [
    {
      "expected": "billing",
      "predicted": "billing",
      "correct": true
    },
    {
      "expected": "billing",
      "predicted": "billing",
      "correct": true
    },
    {
      "expected": "technical",
      "predicted": "technical",
      "correct": true
    },
    {
      "expected": "technical",
      "predicted": "technical",
      "correct": true
    },
    {
      "expected": "account",
      "predicted": "account",
      "correct": true
    },
    {
      "expected": "account",
      "predicted": "other",
      "correct": false
    },
    {
      "expected": "other",
      "predicted": "other",
      "correct": true
    },
    {
      "expected": "other",
      "predicted": "other",
      "correct": true
    }
  ],
  "note": "Small synthetic demo fixture; not a general accuracy estimate."
}


## Troubleshooting and limits

- If the first prediction cannot download weights, check network access to Hugging Face and retry; the package install alone does not cache a checkpoint.
- If model loading runs out of memory, try a machine with more available memory and close other model-heavy processes. Avoid preloading all checkpoints for this small example.
- If you expected generated answers or a conversation, Laya is the wrong abstraction: it returns structured decisions for supplied questions.
- Keep the initial choice set small. Upstream documents limitations for large option sets and cautions against treating shipped confidence as calibrated for your workload.

See the [official Laya documentation](https://github.com/NandhaKishorM/laya) and the project README for installation details, Jev compatibility caveats, and the synthetic evaluation results.